# Rename Brand Values: Old → New Schema

Converts legacy brand names to the canonical 202607+ values across all
`models_*` joblib/CSV files and `results_*` CSV files:

| Old | New |
|-----|-----|
| `Factory US` | `brand outlet` |
| `Specialty US` | `brand us` |
| `Specialty CA` | `brand ca` |

Files whose `brand` column already uses the new values are automatically skipped.

Run cells in order.

In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd

# Add project root so joblib can deserialize custom model classes
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
import functions.lr_models  # noqa: F401

OUTPUT_DIR = PROJECT_ROOT / "data" / "output"

BRAND_MAP = {
    "Factory US":   "brand outlet",
    "Specialty US":  "brand us",
    "Specialty CA":  "brand ca",
}

OLD_BRANDS = set(BRAND_MAP.keys())

print(f"Output dir: {OUTPUT_DIR}")
print(f"Mapping: {BRAND_MAP}")

Output dir: /Users/jeff.parks/Dev/revenue-forecasting/data/output
Mapping: {'Factory US': 'brand outlet', 'Specialty US': 'brand us', 'Specialty CA': 'brand ca'}


## 1. Discovery

Identify which files contain old brand values.

In [2]:
models_to_update = []   # (Path, DataFrame) for joblib files
results_to_update = []  # Path for CSV-only results files

models_skip = []
results_skip = []

# --- models_*.joblib ---
for path in sorted(OUTPUT_DIR.glob("models_*.joblib")):
    df = joblib.load(path)
    if OLD_BRANDS & set(df["brand"].unique()):
        models_to_update.append(path)
    else:
        models_skip.append(path)

# --- results_*.csv ---
for path in sorted(OUTPUT_DIR.glob("results_*.csv")):
    df = pd.read_csv(path)
    if "brand" in df.columns and OLD_BRANDS & set(df["brand"].unique()):
        results_to_update.append(path)
    else:
        results_skip.append(path)

print("MODELS (joblib + CSV)")
print("-" * 55)
for p in models_to_update:
    print(f"  needs update   {p.name}")
for p in models_skip:
    print(f"  already new    {p.name}")

print()
print("RESULTS (CSV only)")
print("-" * 55)
for p in results_to_update:
    print(f"  needs update   {p.name}")
for p in results_skip:
    print(f"  already new    {p.name}")

print(f"\nSummary: {len(models_to_update)} model file(s), {len(results_to_update)} results file(s) to update.")

MODELS (joblib + CSV)
-------------------------------------------------------
  needs update   models_202603_3P.joblib
  needs update   models_202603_reflows.joblib
  needs update   models_202604_1P.joblib
  needs update   models_202604_2P.joblib
  needs update   models_202604_3P.joblib
  needs update   models_202604_BRFS_flat.joblib
  needs update   models_202604_mmm.joblib
  needs update   models_202605_1P.joblib
  needs update   models_202605_2P.joblib
  needs update   models_202605_3P.joblib
  needs update   models_202606_1P.joblib
  already new    models_202606_2P.joblib
  already new    models_202607_1P.joblib
  already new    models_202607_2P.joblib

RESULTS (CSV only)
-------------------------------------------------------
  needs update   results_202603_3P.csv
  needs update   results_202603_reflows.csv
  needs update   results_202604_1P.csv
  needs update   results_202604_1P_marchnew.csv
  needs update   results_202604_1P_marchorig.csv
  needs update   results_202604_2P.csv
 

## 2. Preview (dry run)

Show the brand transformation for one models file and one results file. Nothing is written.

In [3]:
def show_brand_preview(name, old_brands):
    rows = [
        {"Before": b, "After": BRAND_MAP.get(b, f"⚠ UNMAPPED: {b}")}
        for b in sorted(old_brands)
    ]
    print(f"\n{name}")
    display(pd.DataFrame(rows))

if models_to_update:
    df = joblib.load(models_to_update[0])
    show_brand_preview(models_to_update[0].name, df["brand"].unique())

if results_to_update:
    df = pd.read_csv(results_to_update[0])
    show_brand_preview(results_to_update[0].name, df["brand"].unique())


models_202603_3P.joblib


,Before,After
0,Factory US,brand outlet
1,Specialty CA,brand ca
2,Specialty US,brand us



results_202603_3P.csv


,Before,After
0,Factory US,brand outlet
1,Specialty CA,brand ca
2,Specialty US,brand us


## 3. Apply — Models (joblib + CSV)

Update `brand` values in each models joblib and regenerate its companion CSV.

In [4]:
if not models_to_update:
    print("Nothing to update in models files.")
else:
    for joblib_path in models_to_update:
        df = joblib.load(joblib_path)
        df["brand"] = df["brand"].map(lambda v: BRAND_MAP.get(v, v))

        joblib.dump(df, joblib_path)

        csv_path = joblib_path.with_suffix(".csv")
        df.drop(columns=["model_obj"], errors="ignore").to_csv(csv_path, index=False)

        print(f"✓  {joblib_path.name}")

    print(f"\nUpdated {len(models_to_update)} models file(s).")

✓  models_202603_3P.joblib
✓  models_202603_reflows.joblib
✓  models_202604_1P.joblib
✓  models_202604_2P.joblib
✓  models_202604_3P.joblib
✓  models_202604_BRFS_flat.joblib
✓  models_202604_mmm.joblib
✓  models_202605_1P.joblib
✓  models_202605_2P.joblib
✓  models_202605_3P.joblib
✓  models_202606_1P.joblib

Updated 11 models file(s).


## 4. Apply — Results (CSV only)

In [5]:
if not results_to_update:
    print("Nothing to update in results files.")
else:
    for csv_path in results_to_update:
        df = pd.read_csv(csv_path)
        df["brand"] = df["brand"].map(lambda v: BRAND_MAP.get(v, v))
        df.to_csv(csv_path, index=False)
        print(f"✓  {csv_path.name}")

    print(f"\nUpdated {len(results_to_update)} results file(s).")

✓  results_202603_3P.csv
✓  results_202603_reflows.csv
✓  results_202604_1P.csv
✓  results_202604_1P_marchnew.csv
✓  results_202604_1P_marchorig.csv
✓  results_202604_2P.csv
✓  results_202604_3P.csv
✓  results_202604_BRFS_flat.csv
✓  results_202604_mmm.csv
✓  results_202605_1P.csv
✓  results_202605_2P.csv
✓  results_202605_3P.csv
✓  results_202606_1P.csv

Updated 13 results file(s).


## 5. Verify

Reload all files and confirm no old brand names remain.

In [6]:
issues = []

for path in sorted(OUTPUT_DIR.glob("models_*.joblib")):
    df = joblib.load(path)
    stale = OLD_BRANDS & set(df["brand"].unique())
    if stale:
        issues.append((path.name, stale))

for path in sorted(OUTPUT_DIR.glob("results_*.csv")):
    df = pd.read_csv(path)
    if "brand" in df.columns:
        stale = OLD_BRANDS & set(df["brand"].unique())
        if stale:
            issues.append((path.name, stale))

if issues:
    print("⚠ Old brand names still present:")
    for name, vals in issues:
        print(f"  {name}: {vals}")
else:
    all_brands = set()
    for path in OUTPUT_DIR.glob("models_*.joblib"):
        df = joblib.load(path)
        all_brands.update(df["brand"].unique())
    for path in OUTPUT_DIR.glob("results_*.csv"):
        df = pd.read_csv(path)
        if "brand" in df.columns:
            all_brands.update(df["brand"].unique())

    total = (
        len(list(OUTPUT_DIR.glob("models_*.joblib")))
        + len(list(OUTPUT_DIR.glob("results_*.csv")))
    )
    print(f"✓ All {total} file(s) verified — no legacy brand names remain.")
    print(f"\nFinal unique brand values across all files:")
    for b in sorted(all_brands):
        print(f"  {b}")

✓ All 30 file(s) verified — no legacy brand names remain.

Final unique brand values across all files:
  brand us
  brand ca
  brand outlet
